# cv-labins.ipynb

This file uses Gemini- ... to extract plat diagrams, diagram features, and text from relevant LABINS files from 1840-1860.

## Test

I use a file from 1826 to test the method

In [2]:
# load key
import os
from dotenv import load_dotenv
from google import genai

load_dotenv()
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
MODEL = "gemini-3.8-flash"

In [3]:
# load pdf
import base64
from pathlib import Path

PDF_PATH = Path("data/004D07-2.pdf")
pdf_b64 = base64.b64encode(PDF_PATH.read_bytes()).decode()  # batch inline needs base64
print(PDF_PATH.name, f"{len(pdf_b64) / 1e6:.1f} MB b64")  # inline batch cap is 20 MB total

004D07-2.pdf 0.1 MB b64


In [8]:
# prompts
SCHEMA = """json object with these keys:
- doc_type: string
- header: list of {key, value, uncertain}, for township, range, surveyor, year, margin notes
- points: list of {id, x, y}, x and y as fractions of the page from the top left
- segments: list of {id, from, to, label, value, unit, uncertain}
- regions: list of {label, bounded_by: [segment ids]}
- annotations: list of {text, x, y, uncertain}
- uncertain: list of ids or keys you are unsure about
"""

RULES = """rules:
- copy text and numbers exactly as written, in label. put the parsed number in value
- if a reading is doubtful, set uncertain to true. never guess silently
- use null for anything unreadable or blank
- do not add anything that is not on the page
"""

DRAW = """drawing rules:
- ascii only, max 80 columns by 40 rows, north up
- first line: scale as 'scale: 1 char = N units', or '?' if unknown
- lines use - and |, corners and junctions use +
- write each label inline on its line segment, exactly as given
- regions keep their own labels
- unreadable parts are drawn with ?
"""

P1 = f"""this is a survey plat. transcribe it and redraw it as an ascii drawing.
return one json object with two keys:
- data: the json described below
- diagram: the ascii drawing as a string

{SCHEMA}
{RULES}
{DRAW}"""

P2 = f"""this is a survey plat. transcribe it.
return json only, no other text.

{SCHEMA}
{RULES}"""

P3 = f"""this is a survey plat. redraw it as an ascii drawing.
return the drawing only, no other text.

{DRAW}"""

def p4(json_text):
    return f"""this json describes a survey plat. redraw it as an ascii drawing using only the json.
return the drawing only, no other text.

{DRAW}
json:
{json_text}"""

In [9]:
# request helpers
import json, time
from pathlib import Path
from google.genai import errors

OUT = Path("data/labins-detections")
OUT.mkdir(parents=True, exist_ok=True)

def make_request(prompt, pdf=True, as_json=False):
    parts = [{"text": prompt}]
    if pdf:
        parts.append({"inline_data": {"mime_type": "application/pdf", "data": pdf_b64}})
    cfg = {"temperature": 0}
    if as_json:
        cfg["response_mime_type"] = "application/json"
    return {"contents": [{"role": "user", "parts": parts}], "config": cfg}

def ask(req, tries=4):
    for i in range(tries):
        try:
            r = client.models.generate_content(model=MODEL, contents=req["contents"], config=req["config"])
            if not r.text:  # blocked or empty
                return None, "empty response"
            return r.text, None
        except errors.APIError as e:
            if e.code in (429, 500, 503) and i < tries - 1:  # transient
                time.sleep(2 ** i * 5)
                continue
            return None, f"{e.code} {e.message}"  # 400 and 404 don't retry
    return None, "retries exhausted"

results, fails = {}, {}

def step(name, ext, req, is_json=False):
    text, err = ask(req)
    if err is None:
        (OUT / f"{PDF_PATH.stem}.{name}.{ext}").write_text(text)  # save raw first
        if is_json:
            try:
                json.loads(text)
            except json.JSONDecodeError as e:
                err = f"bad json: {e}"
    if err:
        fails[name] = err
    else:
        results[name] = text

In [10]:
# run queries
step("q1", "json", make_request(P1, as_json=True), is_json=True)
step("q2", "json", make_request(P2, as_json=True), is_json=True)
step("q3", "txt", make_request(P3))

# q4 uses q2 json only, no pdf
if "q2" in results:
    step("q4", "txt", make_request(p4(results["q2"]), pdf=False))
else:
    fails["q4"] = "skipped, no q2 json"

# show
if "q1" in results:
    q1 = json.loads(results["q1"])
    print("q1 data\n" + json.dumps(q1.get("data"), indent=2))
    print("\nq1 diagram\n" + str(q1.get("diagram")))
if "q2" in results:
    print("\nq2 json\n" + json.dumps(json.loads(results["q2"]), indent=2))
for name in ["q3", "q4"]:
    print(f"\n{name}\n" + results.get(name, f"FAILED: {fails.get(name)}"))
print("\nfails:", fails)

q1 data
{
  "doc_type": "survey plat",
  "header": [
    {
      "key": "Township",
      "value": "2 N.",
      "uncertain": false
    },
    {
      "key": "Range",
      "value": "7 E.",
      "uncertain": true
    },
    {
      "key": "Surveyor",
      "value": "H. B. Drake",
      "uncertain": false
    },
    {
      "key": "Year",
      "value": "1826",
      "uncertain": false
    },
    {
      "key": "Notes",
      "value": "No variation given.",
      "uncertain": false
    },
    {
      "key": "Page number",
      "value": "7 1/2",
      "uncertain": false
    }
  ],
  "points": [
    {
      "id": "p_0_0",
      "x": 0.23,
      "y": 0.175
    },
    {
      "id": "p_0_1",
      "x": 0.323,
      "y": 0.177
    },
    {
      "id": "p_0_2",
      "x": 0.414,
      "y": 0.179
    },
    {
      "id": "p_0_3",
      "x": 0.506,
      "y": 0.181
    },
    {
      "id": "p_0_4",
      "x": 0.598,
      "y": 0.183
    },
    {
      "id": "p_0_5",
      "x": 0.69,
      "y":